# Introduction to Social Simulation

Statistical analysis benefits primarily the study of variables' distributions. We can complement that approach if we focus on the actors that produce the variables.

A social outcome is a **complex** aggregate of individual actors - what we usually call an **emergent** result of individual decisions. Representing the actor, rather than just the outcome, is what agent-based modeling is about.

# A simple game as an example

Rock, Paper, Scissors is a simultaneous, zero-sum game with three possible outcomes: a draw, a win, or a loss.

- **Rock** beats **Scissors**
- **Paper** beats **Rock**
- **Scissors** beats **Paper**
- Same move on both sides: a draw

Let's represent the game.

## Strategies

Strategies are the options available:

In [1]:
strategies = ['Rock', 'Paper', 'Scissors']

## Rules

The rules tell you that, according to the strategy followed, players get a pay-off:

In [2]:
payoff = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

## Creating and setting up agents

Players have a name, but have no score, and no strategy yet.

In [3]:
Players = [{'name': 'Ava', 'score': 0, 'strategy': None},
           {'name': 'Ben', 'score': 0, 'strategy': None}]

Players

[{'name': 'Ava', 'score': 0, 'strategy': None},
 {'name': 'Ben', 'score': 0, 'strategy': None}]

## Decision-making process

This is the process to choose a strategy: pick one at random.

In [4]:
from random import choice

# simplest strategy: choose randomly
choice(strategies)

'Scissors'

## The moment of truth

**Each agent decides a strategy:**

In [5]:
Players[0]['strategy'] = choice(strategies)

In [6]:
Players[1]['strategy'] = choice(strategies)

**Decisions made:**

In [7]:
Players[0]['strategy'], Players[1]['strategy']

('Scissors', 'Paper')

**Social result of the individual decisions** - the payoff table is looked up once, using both strategies together:

In [8]:
result = payoff[Players[0]['strategy'], Players[1]['strategy']]
result

(1, 0)

**Each agent benefits or suffers from the decision made:**

In [9]:
Players[0]['score'] += result[0]

In [10]:
Players[1]['score'] += result[1]

**Current agent situation:**

In [11]:
Players

[{'name': 'Ava', 'score': 1, 'strategy': 'Scissors'},
 {'name': 'Ben', 'score': 0, 'strategy': 'Paper'}]

**Social outcome** - only now, at the end, do we look at the agents as a table:

In [12]:
import pandas as pd

socialResults = pd.DataFrame(Players)
socialResults

,name,score,strategy
0,Ava,1,Scissors
1,Ben,0,Paper


In [13]:
winnerScore = socialResults.score.max()

# social outcome
socialResults[socialResults.score == winnerScore]

,name,score,strategy
0,Ava,1,Scissors


Notice the order here: the dictionaries came first, and the table came *last*, built from them (`pd.DataFrame(Players)`). The table is a snapshot of the agents at the end - not where the agents came from. Nothing about `Players[0]` or `Players[1]` was ever read out of a spreadsheet or a CSV; every field was set by the code above, one line at a time.

# More players

The same two-dictionary game, just with more agents.

In [14]:
names = ['Ava', 'Ben', 'Cleo', 'Dan']

In [15]:
society = [{'name': n, 'score': 0, 'strategy': None} for n in names]
society

[{'name': 'Ava', 'score': 0, 'strategy': None},
 {'name': 'Ben', 'score': 0, 'strategy': None},
 {'name': 'Cleo', 'score': 0, 'strategy': None},
 {'name': 'Dan', 'score': 0, 'strategy': None}]

## Pairing everyone up

`itertools.combinations` gives every possible pair, once each. A pair is a tuple of two dictionaries:

In [16]:
import itertools

for pair in itertools.combinations(society, 2):
    print(pair)

({'name': 'Ava', 'score': 0, 'strategy': None}, {'name': 'Ben', 'score': 0, 'strategy': None})
({'name': 'Ava', 'score': 0, 'strategy': None}, {'name': 'Cleo', 'score': 0, 'strategy': None})
({'name': 'Ava', 'score': 0, 'strategy': None}, {'name': 'Dan', 'score': 0, 'strategy': None})
({'name': 'Ben', 'score': 0, 'strategy': None}, {'name': 'Cleo', 'score': 0, 'strategy': None})
({'name': 'Ben', 'score': 0, 'strategy': None}, {'name': 'Dan', 'score': 0, 'strategy': None})
({'name': 'Cleo', 'score': 0, 'strategy': None}, {'name': 'Dan', 'score': 0, 'strategy': None})


We usually unpack the pair into two names right away, so each side is easy to work with on its own:

In [17]:
for player1, player2 in itertools.combinations(society, 2):
    print(player1, player2)

{'name': 'Ava', 'score': 0, 'strategy': None} {'name': 'Ben', 'score': 0, 'strategy': None}
{'name': 'Ava', 'score': 0, 'strategy': None} {'name': 'Cleo', 'score': 0, 'strategy': None}
{'name': 'Ava', 'score': 0, 'strategy': None} {'name': 'Dan', 'score': 0, 'strategy': None}
{'name': 'Ben', 'score': 0, 'strategy': None} {'name': 'Cleo', 'score': 0, 'strategy': None}
{'name': 'Ben', 'score': 0, 'strategy': None} {'name': 'Dan', 'score': 0, 'strategy': None}
{'name': 'Cleo', 'score': 0, 'strategy': None} {'name': 'Dan', 'score': 0, 'strategy': None}


## Running a full tournament

We reset `society`, then run several rounds. In each round, every player meets every other player once. This time we print every game as it's played, so the process stays visible, not just the final scores.

In [18]:
# resetting society
society = [{'name': n, 'score': 0, 'strategy': None} for n in names]

# several rounds
for aRound in range(1, 21):
    print(f'--- Round {aRound} ---')

    # in each round:
    for player1, player2 in itertools.combinations(society, 2):
        # each chooses a strategy
        player1['strategy'] = choice(strategies)
        player2['strategy'] = choice(strategies)

        # result from the strategies chosen
        result = payoff[player1['strategy'], player2['strategy']]

        # update scores
        player1['score'] += result[0]
        player2['score'] += result[1]

        print(f"  {player1['name']} ({player1['strategy']}) vs {player2['name']} ({player2['strategy']}): "
              f"{player1['name']} +{result[0]}, {player2['name']} +{result[1]}")

--- Round 1 ---
  Ava (Paper) vs Ben (Scissors): Ava +0, Ben +1
  Ava (Scissors) vs Cleo (Scissors): Ava +0, Cleo +0
  Ava (Paper) vs Dan (Paper): Ava +0, Dan +0
  Ben (Rock) vs Cleo (Scissors): Ben +1, Cleo +0
  Ben (Paper) vs Dan (Paper): Ben +0, Dan +0
  Cleo (Rock) vs Dan (Scissors): Cleo +1, Dan +0
--- Round 2 ---
  Ava (Paper) vs Ben (Rock): Ava +1, Ben +0
  Ava (Rock) vs Cleo (Scissors): Ava +1, Cleo +0
  Ava (Paper) vs Dan (Rock): Ava +1, Dan +0
  Ben (Paper) vs Cleo (Rock): Ben +1, Cleo +0
  Ben (Paper) vs Dan (Scissors): Ben +0, Dan +1
  Cleo (Rock) vs Dan (Scissors): Cleo +1, Dan +0
--- Round 3 ---
  Ava (Scissors) vs Ben (Scissors): Ava +0, Ben +0
  Ava (Paper) vs Cleo (Scissors): Ava +0, Cleo +1
  Ava (Rock) vs Dan (Rock): Ava +0, Dan +0
  Ben (Scissors) vs Cleo (Rock): Ben +0, Cleo +1
  Ben (Scissors) vs Dan (Scissors): Ben +0, Dan +0
  Cleo (Paper) vs Dan (Scissors): Cleo +0, Dan +1
--- Round 4 ---
  Ava (Scissors) vs Ben (Paper): Ava +1, Ben +0
  Ava (Rock) vs Cleo (Pap

## Final situation

In [19]:
society

[{'name': 'Ava', 'score': 21, 'strategy': 'Paper'},
 {'name': 'Ben', 'score': 23, 'strategy': 'Scissors'},
 {'name': 'Cleo', 'score': 17, 'strategy': 'Rock'},
 {'name': 'Dan', 'score': 19, 'strategy': 'Rock'}]

In [20]:
# as a data frame - again, built from the agents, at the very end
socialResults = pd.DataFrame(society)
socialResults

,name,score,strategy
0,Ava,21,Paper
1,Ben,23,Scissors
2,Cleo,17,Rock
3,Dan,19,Rock


In [21]:
winnerScore = socialResults.score.max()

# social outcome
socialResults[socialResults.score == winnerScore]

,name,score,strategy
1,Ben,23,Scissors


### What to notice

Every game was decided by `choice(strategies)` - plain, uniform randomness, the same mechanism the very first two-player demo used. Nothing here assumed anything about *why* a player picks Rock over Paper; there was no rule to assign, no covariate to wire up. The table at the end is just a summary of the dictionaries after 20 rounds of that same simple mechanism, run over and over.

That's the whole point of building agents this way: the table is a byproduct of running the mechanism, not something we computed directly - and the mechanism itself is as plain as it can be.